<a href="https://colab.research.google.com/github/mendump/MachineLearning/blob/master/greywolf_opti.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install numpy pandas scikit-learn torch

In [3]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
CSV_PATH = "/content/merged_crop_yield_5000_unique.csv"
TARGET = "yield_t_ha"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
def metrics(y_true, y_pred, label):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{label:18s} MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f}")
    return {"model": label, "MAE": mae, "RMSE": rmse, "R2": r2}


In [4]:
# 1. Load and validate data
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

if TARGET not in df.columns:
    if {"production_t", "area_ha"}.issubset(df.columns):
        df[TARGET] = df["production_t"] / df["area_ha"].replace(0, np.nan)
    else:
        raise ValueError("CSV needs yield_t_ha or both production_t and area_ha.")

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET, "year"]).copy()
df = df[df[TARGET] >= 0].copy()

In [5]:
# Do not let target leakage enter the predictors.
excluded = {
    TARGET, "production_t", "yield", "yield_tons_per_ha",
    "yield_kg_ha", "yield_t_ha"
}
# Area is retained as a predictor; production is excluded.
feature_cols = [c for c in df.columns if c not in excluded]
if "year" not in feature_cols:
    raise ValueError("A year column is required for temporal splitting.")

In [6]:
# Use later years as a final temporal test set.
years = sorted(df["year"].dropna().unique())
if len(years) < 3:
    raise ValueError("Need at least 3 distinct years for train/validation/test.")
test_start = years[max(1, int(len(years) * 0.8))]
val_start = years[max(1, int(len(years) * 0.65))]

train_df = df[df["year"] < val_start].copy()
val_df = df[(df["year"] >= val_start) & (df["year"] < test_start)].copy()
test_df = df[df["year"] >= test_start].copy()

if min(len(train_df), len(val_df), len(test_df)) == 0:
    # Fallback chronological row split if year coverage is sparse.
    df = df.sort_values("year").reset_index(drop=True)
    n = len(df)
    train_df, val_df, test_df = (
        df.iloc[:int(.65*n)].copy(),
        df.iloc[int(.65*n):int(.80*n)].copy(),
        df.iloc[int(.80*n):].copy()
    )

X_train_raw = train_df[feature_cols]
X_val_raw = val_df[feature_cols]
X_test_raw = test_df[feature_cols]
y_train = train_df[TARGET].astype("float32").to_numpy()
y_val = val_df[TARGET].astype("float32").to_numpy()
y_test = test_df[TARGET].astype("float32").to_numpy()

cat_cols = X_train_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in feature_cols if c not in cat_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
X_val = preprocessor.transform(X_val_raw).astype("float32")
X_test = preprocessor.transform(X_test_raw).astype("float32")


In [7]:
# 2. Conventional baselines
results = []

rf = RandomForestRegressor(
    n_estimators=400, max_depth=None, min_samples_leaf=2,
    random_state=SEED, n_jobs=-1
)
rf.fit(X_train, y_train)
results.append(metrics(y_test, rf.predict(X_test), "Random Forest"))

hgb = HistGradientBoostingRegressor(
    max_iter=300, learning_rate=0.05, max_leaf_nodes=20,
    l2_regularization=1.0, random_state=SEED
)
hgb.fit(X_train, y_train)
results.append(metrics(y_test, hgb.predict(X_test), "HistGradientBoosting"))

Random Forest      MAE=0.4850 | RMSE=0.5873 | R2=0.5017
HistGradientBoosting MAE=0.4790 | RMSE=0.5809 | R2=0.5125


In [8]:
# 3. Grey Wolf Optimizer for feature selection
# Fitness is validation RMSE from a lightweight RF. Test data is never used.
def gwo_feature_selection(Xtr, ytr, Xv, yv, wolves=8, iterations=12):
    dim = Xtr.shape[1]
    rng = np.random.default_rng(SEED)
    positions = rng.uniform(0, 1, size=(wolves, dim))

    def evaluate(pos):
        mask = pos > 0.5
        if mask.sum() == 0:
            mask[np.argmax(pos)] = True
        model = RandomForestRegressor(
            n_estimators=100, min_samples_leaf=2,
            random_state=SEED, n_jobs=-1
        )
        model.fit(Xtr[:, mask], ytr)
        pred = model.predict(Xv[:, mask])
        # Small penalty discourages selecting every feature.
        return np.sqrt(mean_squared_error(yv, pred)) + 0.002 * mask.sum()

    fitness = np.array([evaluate(p) for p in positions])
    order = np.argsort(fitness)
    alpha, beta, delta = positions[order[0]].copy(), positions[order[1]].copy(), positions[order[2]].copy()
    alpha_score = fitness[order[0]]

    for t in range(iterations):
        a = 2 - 2 * (t / max(iterations - 1, 1))
        for i in range(wolves):
            candidates = []
            for leader in (alpha, beta, delta):
                r1, r2 = rng.random(dim), rng.random(dim)
                A = 2 * a * r1 - a
                C = 2 * r2
                D = np.abs(C * leader - positions[i])
                candidates.append(leader - A * D)
            positions[i] = np.clip(np.mean(candidates, axis=0), 0, 1)

        fitness = np.array([evaluate(p) for p in positions])
        order = np.argsort(fitness)
        alpha, beta, delta = (positions[order[j]].copy() for j in range(3))
        alpha_score = fitness[order[0]]
        print(f"GWO iteration {t+1:02d}/{iterations}: validation fitness={alpha_score:.4f}")

    mask = alpha > 0.5
    if mask.sum() == 0:
        mask[np.argmax(alpha)] = True
    return mask


selected = gwo_feature_selection(X_train, y_train, X_val, y_val)
print(f"GWO selected {selected.sum()} of {len(selected)} encoded features.")

GWO iteration 01/12: validation fitness=0.5259
GWO iteration 02/12: validation fitness=0.5320
GWO iteration 03/12: validation fitness=0.5243
GWO iteration 04/12: validation fitness=0.5219
GWO iteration 05/12: validation fitness=0.5209
GWO iteration 06/12: validation fitness=0.5194
GWO iteration 07/12: validation fitness=0.5194
GWO iteration 08/12: validation fitness=0.5194
GWO iteration 09/12: validation fitness=0.5216
GWO iteration 10/12: validation fitness=0.5194
GWO iteration 11/12: validation fitness=0.5194
GWO iteration 12/12: validation fitness=0.5194
GWO selected 10 of 19 encoded features.


In [9]:
# 4. Feature-token Transformer regressor
class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]


class FeatureTokenTransformer(nn.Module):
    def __init__(self, n_features, d_model=64, heads=4, layers=2, dropout=0.15):
        super().__init__()
        self.value_embed = nn.Linear(1, d_model)
        self.feature_embed = nn.Parameter(torch.randn(1, n_features, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=128,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(64, 1)
        )

    def forward(self, x):
        # Each standardized tabular feature becomes one token.
        tokens = self.value_embed(x.unsqueeze(-1)) + self.feature_embed
        z = self.encoder(tokens)
        z = self.norm(z).mean(dim=1)
        return self.head(z).squeeze(-1)


def train_transformer(Xtr, ytr, Xv, yv, epochs=80, lr=1e-3):
    model = FeatureTokenTransformer(Xtr.shape[1]).to(DEVICE)
    train_loader = DataLoader(TabularDataset(Xtr, ytr), batch_size=32, shuffle=True)
    val_loader = DataLoader(TabularDataset(Xv, yv), batch_size=128, shuffle=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    best_state, best_val = None, float("inf")
    patience, wait = 12, 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        pred_all, actual_all = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                pred_all.extend(model(xb.to(DEVICE)).cpu().numpy())
                actual_all.extend(yb.numpy())
        val_rmse = np.sqrt(mean_squared_error(actual_all, pred_all))
        if val_rmse < best_val:
            best_val, wait = val_rmse, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        if wait >= patience:
            break

    model.load_state_dict(best_state)
    return model


Xtr_gwo, Xv_gwo, Xt_gwo = X_train[:, selected], X_val[:, selected], X_test[:, selected]
vit_gwo = train_transformer(Xtr_gwo, y_train, Xv_gwo, y_val)
test_loader = DataLoader(TabularDataset(Xt_gwo), batch_size=128, shuffle=False)
vit_gwo.eval()
predictions = []
with torch.no_grad():
    for xb in test_loader:
        predictions.extend(vit_gwo(xb.to(DEVICE)).cpu().numpy())

results.append(metrics(y_test, np.asarray(predictions), "ViT-GWO"))

pd.DataFrame(results).to_csv("model_comparison_results.csv", index=False)
pd.DataFrame({
    "actual_yield_t_ha": y_test,
    "predicted_yield_t_ha": predictions
}).to_csv("vit_gwo_test_predictions.csv", index=False)

print("\nSaved: model_comparison_results.csv")
print("Saved: vit_gwo_test_predictions.csv")
print("Device:", DEVICE)

ViT-GWO            MAE=0.4256 | RMSE=0.5146 | R2=0.6175

Saved: model_comparison_results.csv
Saved: vit_gwo_test_predictions.csv
Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')